# 02 — Layer 1: individual misregistration detection (RQ1)
Detects items whose noisy label != clean label using individual signals (1 - p_noisy, margin, entropy) and the confident-learning baseline (cleanlab). Reported as AUROC / AUPRC against the clean labels.

In [1]:
# Notebook: 02_layer1_detection
# Shared plotting style: grayscale seaborn, dpi 600, PNG + PDF, no captions.
import os, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"] = "0.2"; plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["font.family"] = "DejaVu Sans"
GREYS = ["#111111", "#555555", "#888888", "#bbbbbb", "#dddddd"]
FIG = os.path.join("..", "results", "figures"); TAB = os.path.join("..", "results", "tables")
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"{name}.{ext}"), dpi=600, bbox_inches="tight")
    plt.close(fig)

import sys; sys.path.append(os.path.join("..", "src"))
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
DATA_DIR = os.path.join("..", "data")
meta = pd.read_parquet(os.path.join(DATA_DIR, "item_meta.parquet"))
pi = np.load(os.path.join(DATA_DIR, "pi_memmap.npy"), mmap_mode="r")
N, K = pi.shape
y = meta["is_misregistered"].values
# clean_id == -1 means the clean label is outside the noisy label space; drop from eval
valid = meta["clean_id"].values >= 0
print(f"N={N:,}  eval items={valid.sum():,}  misregistration rate={y[valid].mean():.3%}")

# --- individual scores ---
rows = np.arange(N)
p_noisy = meta["p_noisy"].values
# margin: p(top) - p(noisy label)
top = np.asarray(pi).max(axis=1)
score_1mp   = 1.0 - p_noisy
score_margin = top - p_noisy
score_ent   = meta["entropy"].values

def report(name, s):
    a = roc_auc_score(y[valid], s[valid]); ap = average_precision_score(y[valid], s[valid])
    print(f"  {name:16s} AUROC={a:.3f}  AUPRC={ap:.3f}"); return a, ap

print("individual signals:")
res = {"1 - p_noisy": report("1 - p_noisy", score_1mp),
       "margin":      report("margin", score_margin),
       "entropy":     report("entropy", score_ent)}

# --- confident learning baseline (cleanlab) ---
try:
    from cleanlab.rank import get_label_quality_scores
    # lower quality => more likely an error; use 1 - quality as the detection score
    q = get_label_quality_scores(labels=meta["noisy_id"].values, pred_probs=np.asarray(pi, dtype=np.float32))
    res["cleanlab (CL)"] = report("cleanlab", 1.0 - q)
except Exception as e:
    print("cleanlab step skipped:", repr(e))

# --- table + figure ---
tab = pd.DataFrame([(k, v[0], v[1]) for k, v in res.items()],
                   columns=["method", "AUROC", "AUPRC"]).sort_values("AUROC", ascending=False)
tab.to_csv(os.path.join(TAB, "t_layer1_detection.csv"), index=False)
print(tab.to_string(index=False))

fig, ax = plt.subplots(figsize=(5.4, 3.4))
x = np.arange(len(tab))
ax.bar(x - 0.2, tab["AUROC"], width=0.4, color=GREYS[0], edgecolor="black", linewidth=0.6, label="AUROC")
ax.bar(x + 0.2, tab["AUPRC"], width=0.4, color=GREYS[2], edgecolor="black", linewidth=0.6, label="AUPRC")
ax.set_xticks(x); ax.set_xticklabels(tab["method"], rotation=30, ha="right")
ax.set_ylabel("score"); ax.legend(frameon=False)
savefig(fig, "f_layer1_detection")
print("saved f_layer1_detection.{png,pdf}")

N=502,310  eval items=502,293  misregistration rate=14.748%
individual signals:
  1 - p_noisy      AUROC=0.526  AUPRC=0.157
  margin           AUROC=0.512  AUPRC=0.151
  entropy          AUROC=0.535  AUPRC=0.157
cleanlab step skipped: ModuleNotFoundError("No module named 'cleanlab'")
     method    AUROC    AUPRC
    entropy 0.534676 0.157245
1 - p_noisy 0.525740 0.156884
     margin 0.511861 0.151228
saved f_layer1_detection.{png,pdf}
